# Logistic regression with TensorFlow

In [1]:
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [2]:
n_rows = 300000
df = pd.read_csv("train.gz", nrows=n_rows)

In [3]:
X = df.drop(['click', 'id', 'hour', 'device_id', 'device_ip'], axis=1).values
Y = df['click'].values

In [4]:
n_train = int(n_rows * 0.9)
X_train = X[:n_train]
Y_train = Y[:n_train].astype('float32')
X_test = X[n_train:]
Y_test = Y[n_train:].astype('float32')

In [5]:
enc = OneHotEncoder(handle_unknown='ignore')
X_train_enc = enc.fit_transform(X_train).toarray().astype('float32')
X_test_enc = enc.transform(X_test).toarray().astype('float32')

In [6]:
batch_size = 1000
train_data = tf.data.Dataset.from_tensor_slices((X_train_enc, Y_train))
train_data = train_data.repeat().shuffle(5000).batch(batch_size).prefetch(1)

In [7]:
n_features = int(X_train_enc.shape[1])
W = tf.Variable(tf.zeros([n_features, 1]))
b = tf.Variable(tf.zeros([1]))

In [8]:
learning_rate = 0.0008
optimizer = tf.optimizers.Adam(learning_rate)

In [9]:
def run_optimization(x, y):
    with tf.GradientTape() as g:
        logits = tf.add(tf.matmul(x, W), b)[:, 0]
        cost = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits))
    gradients = g.gradient(cost, [W, b])
    optimizer.apply_gradients(zip(gradients, [W, b]))

In [10]:
training_steps = 6000
for step, (batch_x, batch_y) in enumerate(train_data.take(training_steps), 1):
    run_optimization(batch_x, batch_y)
    if step % 500 == 0:
        logits = tf.add(tf.matmul(batch_x, W), b)[:, 0]
        loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=batch_y, logits=logits))
        print("step: %i, loss: %f" % (step, loss))

step: 500, loss: 0.423247
step: 1000, loss: 0.422947
step: 1500, loss: 0.409313
step: 2000, loss: 0.392857
step: 2500, loss: 0.411523
step: 3000, loss: 0.406147
step: 3500, loss: 0.374420
step: 4000, loss: 0.386994
step: 4500, loss: 0.438114
step: 5000, loss: 0.399799
step: 5500, loss: 0.403688
step: 6000, loss: 0.383213


In [11]:
logits = tf.add(tf.matmul(X_test_enc, W), b)[:, 0]
pred = tf.nn.sigmoid(logits)
auc_metric = tf.keras.metrics.AUC()
auc_metric.update_state(Y_test, pred)

In [12]:
print(f'AUC on testing set: {auc_metric.result().numpy():.3f}')

AUC on testing set: 0.771
